In [191]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
import numpy as np
import tensorflow as tf

texts = [
    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever",
    "Wouldn't recommend wasting your time",
    "Highly recommended for wasting time"
]

In [192]:
labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])  

vocab_size = 1000
max_len = 6

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen=max_len, padding="post")

print("Word Index:- ")
print(tokenizer.word_index)

print("\n Input Sequences:- \n", X)

Word Index:- 
{'<OOV>': 1, 'this': 2, 'i': 3, 'movie': 4, 'film': 5, 'very': 6, 'acting': 7, 'story': 8, 'wasting': 9, 'time': 10, 'love': 11, 'is': 12, 'amazing': 13, 'good': 14, 'excellent': 15, 'hate': 16, 'terrible': 17, 'boring': 18, 'worst': 19, 'ever': 20, "wouldn't": 21, 'recommend': 22, 'your': 23, 'highly': 24, 'recommended': 25, 'for': 26}

 Input Sequences:- 
 [[ 3 11  2  4  0  0]
 [ 2  5 12 13  0  0]
 [ 6 14  7  0  0  0]
 [15  8  0  0  0  0]
 [ 3 16  2  4  0  0]
 [17  5  0  0  0  0]
 [ 6 18  8  0  0  0]
 [19  7 20  0  0  0]
 [21 22  9 23 10  0]
 [24 25 26  9 10  0]]


In [193]:
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

In [194]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

In [195]:
# build the model
embed_dim = 8
num_heads = 1
ff_dim = 32
inputs = layers.Input(shape=(max_len,))
 
x = TokenAndPositionEmbedding(
    max_length=max_len,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(1, activation="sigmoid")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [196]:
model.compile(
    optimizer="adam",
    loss = "binary_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "functional_41"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_41 (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_20 │ (None, 6, 8)           │         8,048 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_20            │ (None, 6, 8)           │           872 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_20     │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_62 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,929 (34.88 KB)

 Trainable params: 8,929 (34.88 KB)

 Non-trainable params: 0 (0.00 B)

In [197]:
model.fit(X,labels,epochs=30,batch_size=2,verbose=1)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6000 - loss: 0.7611
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 0.7131
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7000 - loss: 0.5945
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7000 - loss: 0.5608
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7000 - loss: 0.5288
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7000 - loss: 0.4840
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8000 - loss: 0.4565
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.4362
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.4108
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.3791
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.3574
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.3319
E

In [198]:
test_sentences = [
    "I love the film",
    "This movie was awful",
    "Rubbish movie",
    "Boring but Meaningful"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_len, padding="post")
predictions = model.predict(test_pad)
 
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction[0])
    if prediction[0] > 0.5:
        print("Prediction: Positive")
    else:
        print("Prediction: Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
I love the film -> 0.8665991
Prediction: Positive
This movie was awful -> 0.13033898
Prediction: Negative
Rubbish movie -> 0.70543534
Prediction: Positive
Boring but Meaningful -> 0.016749265
Prediction: Negative
